# 13.03 · Text-to-speech: Orpheus and Gemini TTS for a support line

**Module 13 · Multimodal & Voice** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2 h |
| 📋 Prerequisites | Lesson 13.02 (audio basics, Whisper). Read the module README first. |
| 🔑 Keys | optional: `GROQ_API_KEY` (Orpheus, Whisper), `GOOGLE_API_KEY` (Gemini TTS, free tier) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Rewrite** a chat answer so it can be spoken: no Markdown or URLs, IDs spelled out, currencies said aloud.
2. **Split** text into Orpheus's 200-character requests at sentence boundaries, with the vocal direction on every chunk.
3. **Guard** a TTS quota before sending (Orpheus free tier: 100 requests and 3.6K tokens a day).
4. **Synthesize** the same answer with Groq Orpheus and Gemini TTS, and compare time to first audio, total time, languages and cost.
5. **Check** generated speech automatically by transcribing it back with Whisper.
6. **Build** a fallback chain with a cache that always ends in a text answer.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Groq Orpheus](https://console.groq.com/docs/text-to-speech/orpheus) | Voices, `[directions]`, 200-character limit, WAV only, $22 per 1M characters |
| [Groq rate limits](https://console.groq.com/docs/rate-limits) | Orpheus free tier: 10 requests/min, 100/day, 1.2K tokens/min, 3.6K/day |
| [Gemini speech generation](https://ai.google.dev/gemini-api/docs/speech-generation) | Interactions API, `speech_metadata.style`, 30 voices, 100+ languages, streaming PCM |
| [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing) | TTS models free of charge on the free tier (content may be used to improve Google's products) |

> **Why this matters at work.** Phone support, accessibility (reading a page aloud), in-car and kiosk assistants all end with a voice. A voice that reads `**C-1002**` as "star star C minus one thousand and two" sounds broken, and a TTS quota that runs out at 11 a.m. means silence. This lesson builds the speaking half of Project 13's support line.

> 🔑 **One-time step for Orpheus:** Groq requires an org admin to **accept Orpheus's terms** before the API serves it; until then every request fails with `400 model_terms_required`. Open the [Groq playground for Orpheus](https://console.groq.com/playground?model=canopylabs%2Forpheus-v1-english), accept the terms, and you're done. If you skip it, the Orpheus cells say so and the lesson continues with Gemini and a simulated speaker.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "groq>=0.37.1,<1" "google-genai>=2.25.0,<3" "numpy>=2.3.0,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("groq", "google-genai", "numpy",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/13-multimodal-and-voice/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'GOOGLE_API_KEY')
LANGSMITH_PROJECT = 'm13-text-to-speech'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_GOOGLE_KEY = is_set("GOOGLE_API_KEY") or is_set("GEMINI_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m13-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("vision    :", settings.model_vision, "(Groq, preview)")
print("multimodal:", settings.model_multimodal)
print("stt / tts :", settings.model_stt, "/", settings.tts_provider)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_GOOGLE_KEY:
    print("⚠️ No GOOGLE_API_KEY: 💸 Gemini cells will print ⏭️ Skipped (free key: https://aistudio.google.com/apikey).")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models (qwen3.8 vision, Orpheus) warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from IPython.display import Audio, display

AUDIO_DIR = REPO_ROOT / "data" / "outputs" / "m13" / "tts"   # git-ignored
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = []   # one row per synthesis, compared in Part 8

# The answer the support agent wrote in chat (Markdown, an ID, a price, a URL).
ANSWER = ("Your **Nimbus Pro** plan for customer `C-1002` costs US$349.00 per month, billed on the 5th. "
          "See https://orbita.example/billing for details. If the export still fails with error 503, restart the "
          "sync agent from Settings, then try again; if it still fails, reply to this message and we will call you "
          "within one business hour.")
ANSWER_ES = ("Tu plan Nimbus Pro para el cliente C-1-0-0-2 cuesta 349 dólares al mes. Si la exportación sigue fallando, "
             "reinicia el agente de sincronización desde Configuración y vuelve a intentarlo.")
print(len(ANSWER), "characters in the chat answer")

---
## Part 1 · Text a voice can read

Chat answers are written for eyes. Before TTS: drop Markdown and code marks, replace URLs (nobody follows a URL read aloud), spell IDs character by character (Orpheus reads `2-0-3` digit by digit, but `203` as "two hundred and three"), and say currencies the way a person would.

In [ ]:
# 🧪 Offline: chat text → speakable text.
from mastery.voice import normalize_for_speech

SPOKEN = normalize_for_speech(ANSWER)
print("chat   :", ANSWER)
print("spoken :", SPOKEN)

**You should now see** the spoken version: `Your Nimbus Pro plan for customer C-1-0-0-2 costs 349 dollars per month, …  See the link in our message for details. …`

### What just happened?

This step is plain string processing, and it decides whether the voice sounds competent. Grow the rules from real failures: listen to a sample of answers each week and add a rule for every word the voice gets wrong. An alternative is to ask the agent for a separate `spoken` field in its structured output (Exercise 3).

---
## Part 2 · 200 characters per request, and a quota that runs out

Orpheus accepts at most **200 characters** per request. `split_for_tts` cuts at sentence ends first, then clauses, then words, and repeats the vocal direction on every chunk (each request is synthesized on its own). Then the quota: the free tier allows 100 Orpheus requests and **3.6K tokens a day**.

In [ ]:
# 🧪 Offline: chunks, and how many such answers fit in a free-tier day.
from mastery.voice import BudgetExceeded, TtsBudget, split_for_tts

chunks = split_for_tts(SPOKEN, direction=None)
for c in chunks:
    print(f"{len(c):3d}  {c}")

budget = TtsBudget()          # Orpheus free tier; tokens_per_char=1.0 is a cautious guess (Groq doesn't say)
answers = 0
try:
    while True:
        budget.check(chunks)
        for c in chunks:
            budget.record(c)
        answers += 1
        budget.clock = (lambda t=time.time() + 61 * answers: t)   # pretend a minute passes between answers
except BudgetExceeded as stop:
    print(f"\n{answers} answers like this fit in one day; then: {stop}")

**You should now see** two chunks (about 130 and 190 characters), then **11 answers** per day before `daily TTS budget: … tokens used`.

### What just happened?

Eleven spoken answers a day is a demo, not a support line. The guard turns a surprise `429` at 11 a.m. into a decision you make in advance: cap Orpheus for short phrases, switch to another provider (Part 4), or pay. Measure the real token count per character from Groq's `x-ratelimit-remaining-tokens` header and set `tokens_per_char`; the 1.0 here is deliberately pessimistic.

---
## Part 3 · Groq Orpheus (preview)

`OrpheusSpeaker` splits, checks the budget, sends one request per chunk and stitches the WAVs. **Time to first audio** is what a caller notices: a player can start as soon as chunk 1 arrives (Lesson 13.04 does that).

In [ ]:
# 💸 Orpheus: the spoken answer in 200-character chunks, voice troy, no direction (natural support tone).
from mastery.voice import FakeSpeaker, ModelTermsRequired, OrpheusSpeaker

orpheus_budget = TtsBudget()
ORPHEUS_OK = HAVE_GROQ_KEY
if HAVE_GROQ_KEY:
    orpheus = OrpheusSpeaker(budget=orpheus_budget)
else:
    orpheus = FakeSpeaker("simulated-orpheus", budget=orpheus_budget)
    print("⏭️ Skipped: GROQ_API_KEY is not set; SIMULATED speaker (a soft tone per chunk).")
try:
    said = orpheus.speak(SPOKEN)
except ModelTermsRequired as error:      # 400 model_terms_required: accept the terms once (see Step 0)
    ORPHEUS_OK = False
    print("⚠️ Orpheus is not enabled for your Groq org yet:", error)
    print("   This lesson continues with a SIMULATED speaker; Gemini and the fallback chain still run.")
    orpheus = FakeSpeaker("simulated-orpheus", budget=orpheus_budget)
    said = orpheus.speak(SPOKEN)
if not said.simulated:                                   # Part 6 transcribes this file
    (AUDIO_DIR / "answer-orpheus.wav").write_bytes(said.wav)
print(f"{said.provider}: {said.chunks} requests, first audio after {said.first_audio_s:.2f} s, all after {said.total_s:.2f} s, "
      f"${said.cost_usd:.5f}; budget left {orpheus_budget.remaining()}")
RESULTS.append({"provider": said.provider, "lang": "en", "requests": said.chunks, "first audio s": said.first_audio_s,
                "total s": said.total_s, "cost $": f"{said.cost_usd:.5f}", "simulated": said.simulated})
display(Audio(said.wav))

> 💸 Calls Orpheus twice (about 320 characters, $0.007; 2 of your 100 free requests a day). Without `GROQ_API_KEY` it prints `⏭️ Skipped` and a tone stands in.

**You should now see** `groq:canopylabs/orpheus-v1-english: 2 requests, first audio after ~0.5 s, all after ~1 s` (live) and a player. Listen for the ID: it should be "C, one, zero, zero, two".

---
## Part 4 · Gemini TTS: one request, any language (free tier)

Gemini TTS has no 200-character limit, speaks 100+ languages (Orpheus English is English only) and takes a `style` for the delivery. On the free tier it costs nothing, and Google may use what you send to improve its products: fine for this synthetic answer, not for real customer data.

In [ ]:
# 💸 Gemini TTS: the English answer in one request, then the Spanish one.
from mastery.voice import GeminiSpeaker

if HAVE_GOOGLE_KEY:
    gemini = GeminiSpeaker(style="calm, friendly support agent")
    for lang, text in (("en", SPOKEN), ("es", ANSWER_ES)):
        said_g = gemini.speak(text)
        (AUDIO_DIR / f"answer-gemini-{lang}.wav").write_bytes(said_g.wav)
        print(f"{lang}: {said_g.provider}: {said_g.chunks} request, first audio after {said_g.first_audio_s:.2f} s")
        RESULTS.append({"provider": said_g.provider, "lang": lang, "requests": said_g.chunks, "first audio s": said_g.first_audio_s,
                        "total s": said_g.total_s, "cost $": "0 (free tier)", "simulated": False})
        display(Audio(said_g.wav))
else:
    print("⏭️ Skipped: GOOGLE_API_KEY is not set (see the setup cell).")

> 💸 Calls Gemini TTS twice (free tier). Without `GOOGLE_API_KEY` it prints `⏭️ Skipped`.

**You should now see** two players and first-audio times of one to a few seconds: one request means the caller waits for the **whole** answer (unless you stream, Lesson 13.05), whereas Orpheus's first chunk arrives sooner.

### What just happened?

The two providers trade off differently: Orpheus is fast per chunk but English-only, preview, and tightly capped on the free tier; Gemini is multilingual and generous on the free tier, but a unary request returns only when the whole clip is ready. That is why the course defaults to Gemini (`MASTERY_TTS_PROVIDER=gemini`) and keeps Orpheus as the Groq option.

---
## Part 5 · Directions and style, deliberately

Orpheus reads bracketed directions (`[cheerful]`, `[professionally]`); Gemini 3.8 reads a turn-level `style`. Groq's own advice for **support**: no directions, the natural tone is friendlier than an acted one. Use directions where emotion is the content, such as an apology for an outage.

In [ ]:
# 💸 Orpheus: the same apology with and without a direction (2 requests).
APOLOGY = "We're sorry about this morning's outage. Everything is working again, and this month's invoice will reflect it."
if HAVE_GROQ_KEY and ORPHEUS_OK:
    for direction in (None, "[sincerely apologetic]"):
        spk = OrpheusSpeaker(direction=direction, budget=orpheus_budget)
        clip = spk.speak(APOLOGY)
        print(direction or "(no direction)", f"→ {clip.total_s:.2f} s")
        display(Audio(clip.wav))
    print("budget left:", orpheus_budget.remaining())
elif HAVE_GROQ_KEY:
    print("⏭️ Skipped: Orpheus terms not accepted yet (model_terms_required, see Part 3).")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 Two Orpheus requests (about 230 characters).

**You should now see** two players. The directed one is slower and softer; decide by ear which one you would want to hear after an outage. Unknown directions are ignored silently, so a typo (`[sincerly]`) just sounds neutral.

---
## Part 6 · Check speech automatically: transcribe it back

You can't listen to every generated answer. A cheap automatic check: transcribe the audio with Whisper (Lesson 13.02) and compare it with the text you sent. A high WER means a mispronounced ID, a skipped clause or a garbled chunk boundary.

In [ ]:
# 💸 Whisper round trip on the Orpheus answer (and Gemini's, if it exists).
from mastery.voice import GroqTranscriber, wer

if HAVE_GROQ_KEY:
    whisper = GroqTranscriber()
    for name in ("answer-orpheus.wav", "answer-gemini-en.wav"):
        path = AUDIO_DIR / name
        if path.is_file():
            heard = whisper.transcribe(path, language="en", prompt="Nimbus Pro, Orbita, C-1002")
            print(f"{name:<22} WER {wer(SPOKEN, heard.text):.3f}  heard: {heard.text[:90]}…")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

> 💸 One or two Whisper requests (about 40 seconds of audio).

**You should now see** WER around 0.05–0.15 for each file. Most "errors" are formatting: Whisper writes `C1002` or `$349` where the spoken text says `C-1-0-0-2` and `349 dollars`. Read the heard text for the ID: that is the error that matters.

### What just happened?

That's a regression test for your voice: keep 20 representative answers, synthesize them after every change (voice, provider, normalizer), and alert when WER jumps. It does not judge tone or naturalness; for that, people still listen to a small sample.

---
## Part 7 · Fallbacks and a cache that always ends in text

TTS fails like any API (429 when the day's quota is gone, 5xx, a retired preview model). A voice product must still answer. `speak_with_fallback` tries each speaker in order, caches successes by (speaker, voice, text), and finally returns `wav=None` with the text, so the UI shows the answer instead of an error.

In [ ]:
# 🧪 Offline: a chain where the first provider is down, and one where everything is down.
from mastery.voice import speak_with_fallback

down = FakeSpeaker("primary-tts", fail="429 daily request limit reached")
backup = FakeSpeaker("backup-tts")
for chain in ([down, backup], [down]):
    out = speak_with_fallback(SPOKEN, chain)
    print(f"{' → '.join(s.name for s in chain):<28} answered by {out.provider:<11} audio={'yes' if out.wav else 'no '}  errors={out.errors}")

**You should now see** `answered by backup-tts audio=yes` with the 429 in `errors`, then `answered by text-only audio=no` with the text still returned.

In [ ]:
# 💸 The course chain: Gemini (free tier) → Orpheus → text, with a disk cache; the second call is free.
chain = []
if HAVE_GOOGLE_KEY:
    chain.append(GeminiSpeaker())
if HAVE_GROQ_KEY and ORPHEUS_OK:
    chain.append(OrpheusSpeaker(budget=orpheus_budget))
GREETING = "Thanks for calling Orbita Cloud support. How can I help you today?"
for attempt in (1, 2):
    start = time.perf_counter()
    out = speak_with_fallback(GREETING, chain, cache_dir=AUDIO_DIR / "cache")
    print(f"call {attempt}: {out.provider:<40} cache={out.from_cache}  {time.perf_counter() - start:.3f} s")

> 💸 With keys, one TTS request the first time this greeting is spoken; later calls read the cache. With no key at all, the chain is empty and both calls return `text-only`.

**You should now see** (with `GOOGLE_API_KEY`) `call 1: google_genai:gemini-3.8-flash-lite-tts cache=False` in a second or two, then `call 2: … cache=True` in milliseconds.

### What just happened?

Greetings, menu prompts and hold messages are the same every time: synthesize once, serve from the cache, and the per-character cost and the daily quota stop mattering for them. Only genuinely new answers spend quota.

---
## Part 8 · Compare

In [ ]:
# 🧪 Offline: everything synthesized in this run.
table(RESULTS, ["provider", "lang", "requests", "first audio s", "total s", "cost $", "simulated"])

**You should now see** at least the Orpheus row (simulated without a key) and, with `GOOGLE_API_KEY`, two Gemini rows.

| | Groq Orpheus | Gemini 3.8 Flash-Lite TTS |
|---|---|---|
| Status | Preview | Generally available (check the model page) |
| Languages | English (plus a Saudi Arabic model) | 100+, Spanish included |
| Per request | 200 characters | Whole answers |
| Free tier | 100 requests, 3.6K tokens a day | Free of charge (limits in AI Studio); content may train Google's products |
| Paid | $22 per 1M characters | See the pricing page |
| Delivery control | `[directions]` inline | `speech_metadata.style` + inline `<tags>` |

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `400` from Orpheus | Over 200 characters (direction included) | `split_for_tts` with the direction passed in |
| `400 model_terms_required` | Orpheus's terms not accepted for your org | Org admin accepts once in the [playground](https://console.groq.com/playground?model=canopylabs%2Forpheus-v1-english) (`ModelTermsRequired` says so) |
| IDs read as big numbers | `1002` spoken as "one thousand and two" | `normalize_for_speech` spells `C-1-0-0-2` |
| "star star" or a URL read aloud | Chat Markdown sent to TTS | Normalize first, or ask the agent for a `spoken` field |
| `429` mid-morning | Orpheus free tier: 100 requests / 3.6K tokens a day | `TtsBudget` before sending; Gemini as default; cache fixed phrases |
| Direction in chunk 1 only | Each request is independent | Repeat it on every chunk (`direction=`) |
| Clicks between chunks | Hard joins | `stitch_wavs` adds a short gap; join at sentence ends |
| Spanish text spoken with an English accent | Orpheus English is English only | Gemini TTS for other languages |

## ⚠️ Production notes

> ⚠️ **Production note: TTS is a preview dependency with a hard daily cap.** (Parts 2–3, 7)
> Orpheus is preview: pin it, monitor errors, and keep a fallback that you test in CI. Budget per day, cache everything that repeats, and make text-only a first-class answer, never an error page.

> ⚠️ **Production note: synthetic voices must be disclosed.** (Parts 3–5)
> Tell callers they are talking to an automated assistant. Never imitate a real person's voice without written consent (Gemini's voice replication requires consent audio for this reason).

## 🏋️ Exercises

1. **(Easy)** Pick two other Orpheus voices from `ORPHEUS_VOICES` and two Gemini voices (for example `Puck`, `Charon`) for the greeting. Which would you choose for a support line, and why?
2. **(Medium)** Extend `normalize_for_speech` for Colombian phone numbers (`300 555 0142` → "3 0 0, 5 5 5, 0 1 4 2") and times (`14:30` → "2:30 in the afternoon"). Add a test.
3. **(Medium)** Ask the fast model for an answer with two fields, `text` (for the screen) and `spoken` (for TTS, no Markdown, IDs spelled). Compare its `spoken` with `normalize_for_speech(text)`.
4. **(Stretch)** Make the Part 6 round trip a function `voice_regression(texts, speaker, threshold=0.2)` that returns the texts whose WER is above the threshold, and run it on five answers.

<details>
<summary><b>Show solutions</b></summary>

**1.** Use `speak_with_fallback(GREETING, [OrpheusSpeaker(voice="hannah")])` and `GeminiSpeaker(voice="Charon")`. There is no right answer; a calm, clear voice that is easy to understand on a phone line usually wins over an expressive one.

**2.**
```python
import re

def phone_and_time(text):
    text = re.sub(r"\b(\d{3}) (\d{3}) (\d{4})\b", lambda m: ", ".join(" ".join(g) for g in m.groups()), text)
    def say_time(m):
        h, mnt = int(m.group(1)), m.group(2)
        part = "in the morning" if h < 12 else "in the afternoon" if h < 18 else "in the evening"
        return f"{(h - 1) % 12 + 1}:{mnt} {part}"
    return re.sub(r"\b(\d{1,2}):(\d{2})\b", say_time, text)

assert phone_and_time("Call 300 555 0142 at 14:30") == "Call 3 0 0, 5 5 5, 0 1 4 2 at 2:30 in the afternoon"
```

**3.**
```python
from pydantic import BaseModel
from mastery.models import get_model

class Answer(BaseModel):
    text: str
    spoken: str

ans = get_model("fast").with_structured_output(Answer).invoke(
    "Answer for a customer: what does Nimbus Pro cost for customer C-1002? It's US$349 per month. "
    "'spoken' must have no Markdown or URLs and spell IDs like C-1-0-0-2.")
print(ans.spoken, "|", normalize_for_speech(ans.text))
```

**4.**
```python
def voice_regression(texts, speaker, threshold=0.2):
    stt, bad = GroqTranscriber(), []
    for text in texts:
        heard = stt.transcribe(speaker.speak(text).wav, language="en")
        score = wer(text, heard.text)
        if score > threshold:
            bad.append((round(score, 3), text, heard.text))
    return bad
```
</details>

## 🔎 Docs drill (4 min)

On Groq's Orpheus page, how do you make Orpheus read a number digit by digit, and what do the docs recommend for customer-support voices?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Groq Orpheus](https://console.groq.com/docs/text-to-speech/orpheus) and find the natural-conversation examples.
2. The tip says pure numbers like `203` are read as "two hundred and three"; write `2-0-3` to have them read one by one.
3. For customer support and assistants the page recommends **no directions** (the default conversational cadence).

</details>

## ✅ Checkpoint

- [ ] I turned a chat answer into speakable text.
- [ ] I split text into ≤ 200-character chunks with the direction on every chunk.
- [ ] I computed how many answers fit in Orpheus's free daily quota and guarded it with `TtsBudget`.
- [ ] I synthesized the same answer with Orpheus and Gemini and compared first-audio time, languages and cost.
- [ ] I checked generated speech by transcribing it back.
- [ ] I built a fallback chain with a cache that ends in text.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Re-check [Groq's TTS models](https://console.groq.com/docs/text-to-speech) and [rate limits](https://console.groq.com/docs/rate-limits): is Orpheus still preview, are there new voices or languages, and what are the free-tier TPD and RPD? From one Orpheus response, read the `x-ratelimit-remaining-tokens` header (`client.audio.speech.with_raw_response.create(...)`) before and after a request and log how many tokens a 100-character input used. Log your findings in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Normalize before speaking: no Markdown or URLs, IDs spelled, currencies said.
- Orpheus: 200 characters per request, directions on every chunk, a tight free daily quota; guard it before sending.
- Gemini TTS: whole answers, 100+ languages, free tier (synthetic data only).
- Check speech automatically with a Whisper round trip.
- Fallback chain + cache, always ending in a text answer.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 13-04-voice-agent-sandwich.ipynb** → [`13-04-voice-agent-sandwich.ipynb`](13-04-voice-agent-sandwich.ipynb).

[Module 13 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)